In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 3, Finished, Available, Finished, False)

#### **Reading All Tables**

In [2]:
# Get last run timestamps for each table
def get_last_run(table_name):
    return spark.sql(f"""
        SELECT COALESCE(MAX(run_timestamp), '1900-01-01')
        FROM Olist_Gold_Lakehouse.dbo.gold_data_quality_log
        WHERE table_name = '{table_name}'
        AND status = 'success'
        AND run_type = 'pipeline'
    """).collect()[0][0]

last_run_order_items   = get_last_run('silver_olist_order_items')
last_run_orders        = get_last_run('silver_olist_orders')
last_run_customers     = get_last_run('silver_olist_customers')
last_run_payments      = get_last_run('silver_olist_order_payments')
last_run_reviews       = get_last_run('silver_olist_order_reviews')

# Read and filter only new records
df1 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_customers").filter(col("ingestion_timestamp") >= last_run_customers)
df3 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_order_items").filter(col("ingestion_timestamp") >= last_run_order_items)
df4 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_order_payments").filter(col("ingestion_timestamp") >= last_run_payments)
df5 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_order_reviews").filter(col("ingestion_timestamp") >= last_run_reviews)
df6 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_orders").filter(col("ingestion_timestamp") >= last_run_orders)

# These don't change often so no incremental needed
df2  = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_geolocation")
df7  = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_products")
df8  = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_sellers")
df9  = spark.read.table("Olist_Bronze_Lakehouse.dbo.product_category_name_translation")
df10 = spark.read.table("Olist_Silver_Lakehouse.dbo.ref_country_state")
df11 = spark.read.table("Olist_Silver_Lakehouse.dbo.ref_city_master")


StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 4, Finished, Available, Finished, False)

#### **Transforming Table : olist_customers**

In [3]:
def normalize_column(column):
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column
    
df1_clean = (
    df1
    .withColumn("customer_state_clean", trim(upper(col("customer_state"))))
    .withColumn("customer_city_clean", normalize_column(col("customer_city")))
)
df10_clean = (
    df10
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("state_name_clean", normalize_column(col("state_name")))
)
df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

ref_city = df11_clean.select("city_name_clean").distinct()

ref_state = df10_clean.select("state_code").distinct()

ref_state_city = df11_clean.select(
    "state_code",
    "city_name_clean"
).distinct()

df1_final = (
df1_clean.withColumn(
    "ranking",
    row_number().over(Window.partitionBy("customer_id").orderBy(desc("ingestion_timestamp")))
)
.filter(col("ranking") == 1)
.drop("ranking")
.alias("c") \
.join(
    ref_city.alias("city_ref"),
    col("c.customer_city_clean") == col("city_ref.city_name_clean"),
    "left"
)
.join(
    ref_state.alias("state_ref"),
    col("c.customer_state_clean") == col("state_ref.state_code"),
    "left"
)
.join(
    ref_state_city.alias("sc_ref"),
    (col("c.customer_state_clean") == col("sc_ref.state_code")) &
    (col("c.customer_city_clean") == col("sc_ref.city_name_clean")),
    "left"
)
.withColumn(
    "is_city_valid",
    when(col("city_ref.city_name_clean").isNull(),0)
    .otherwise(1)
)
.withColumn(
    "is_state_valid",
    when(col("state_ref.state_code").isNull(),0)
    .otherwise(1)
)
.withColumn(
    "is_state_city_valid",
    when(col("sc_ref.state_code").isNull(),0)
    .otherwise(1)
)
.select(
    col("c.customer_id"),
    col("c.customer_unique_id"),
    col("c.customer_zip_code_prefix"),
    col("c.customer_city_clean").alias("customer_city"),
    col("c.customer_state_clean").alias("customer_state"),
    "is_city_valid",
    "is_state_valid",
    "is_state_city_valid",
    "ingestion_timestamp"
    )
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 5, Finished, Available, Finished, False)

In [4]:
from delta.tables import DeltaTable

target_table = "Olist_Silver_Lakehouse.dbo.silver_olist_customers"

if spark.catalog.tableExists(target_table):

    delta_table = DeltaTable.forName(spark, target_table)

    delta_table.alias("t").merge(
        df1_final.alias("s"),
        "t.customer_id = s.customer_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    df1_final.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 6, Finished, Available, Finished, False)

#### **Transforming Table : olist_orders**

In [5]:
df6_final = (
        df6.withColumn(
        "ranking",
        row_number().over(Window.partitionBy("order_id")
        .orderBy(
        desc(col("order_delivered_customer_date").isNotNull()),
        desc(col("order_approved_at").isNotNull()),
        desc("order_purchase_timestamp")
         )
        )
    )
    .join(df1.select("customer_id", "customer_unique_id"),
    on="customer_id",
    how="left"
    )
    .withColumn(
    "customer_unique_id",
    coalesce(col("customer_unique_id"), lit("-1"))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")

    .withColumn(
        "is_purchase_after_approved",
        when(
            (col("order_purchase_timestamp").isNotNull()) &
            (col("order_approved_at").isNotNull()) &
            (col("order_purchase_timestamp") > col("order_approved_at")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_approval_after_carrier",
        when(
            (col("order_delivered_carrier_date").isNotNull()) &
            (col("order_approved_at").isNotNull()) &
            (col("order_approved_at") > col("order_delivered_carrier_date")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_carrier_after_customer",
        when(
            (col("order_delivered_carrier_date").isNotNull()) &
            (col("order_delivered_customer_date").isNotNull()) &
            (col("order_delivered_carrier_date") > col("order_delivered_customer_date")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_late_delivery",
        when(
            (col("order_status") == "delivered") &
            (col("order_delivered_customer_date").isNotNull()) &
            (col("order_estimated_delivery_date").isNotNull()) &
            (col("order_delivered_customer_date") > col("order_estimated_delivery_date")),
            1
            )
        .otherwise(0)
    )
    
    .select("order_id",
            "customer_id",
            "customer_unique_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "is_purchase_after_approved",
            "is_approval_after_carrier",
            "is_carrier_after_customer",
            "is_late_delivery")
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 7, Finished, Available, Finished, False)

In [6]:
from delta.tables import DeltaTable

target_table = "Olist_Silver_Lakehouse.dbo.silver_olist_orders"

if spark.catalog.tableExists(target_table):

    delta_table = DeltaTable.forName(spark, target_table)

    delta_table.alias("t").merge(
        df6_final.alias("s"),
        "t.order_id = s.order_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    df6_final.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 8, Finished, Available, Finished, False)

#### **Transforming Table : olist_order_items**

In [7]:
df_silver_orders = spark.read.table("Olist_Silver_Lakehouse.dbo.silver_olist_orders")

df3_final = (
    df3.alias("oi").withColumn(
        "ranking",
        row_number().over(Window.partitionBy("order_id","order_item_id")
        .orderBy(
            col("ingestion_timestamp").desc()  
            ))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")
    .join(df_silver_orders.alias("o"),"order_id","left")
    .withColumn(
    "is_delivered_price_missing",
    when((col("oi.price").isNull()) & (col("o.order_status") == "delivered"),1)
    .otherwise(0)
    )
    .withColumn(
    "is_price_missing",
    when(col("oi.price").isNull(),1)
    .otherwise(0)
    )
    .withColumn(
    "is_price_valid",
    when((col("oi.price") >= 0) & (col("oi.price").isNotNull()),1)
    .otherwise(0)
    )
    .withColumn(
        "is_refunded",
        when(col("oi.price") < 0, 1)
        .otherwise(0)
    )
     .withColumn(
        "is_delivered_freight_value_missing",
        when((col("oi.freight_value").isNull()) & (col("o.order_status") == "delivered" ),1)
        .otherwise(0)
    )
    .withColumn(
        "is_freight_value_missing",
        when(col("oi.freight_value").isNull(),1)
        .otherwise(0)
    )
    .withColumn(
        "is_freight_value_valid",
        when((col("oi.freight_value") >= 0) & (col("oi.freight_value").isNotNull()),1)
        .otherwise(0)
    )
    .withColumn(
    "is_invalid_canceled_delivery",
    when(
        (col("o.order_status") == "canceled") & 
        col("order_delivered_customer_date").isNotNull(),
        1
    ).otherwise(0)
    )
    .withColumn(
    "is_order_date_valid",
    when(
        (
        (col("order_approved_at").isNull() | col("order_purchase_timestamp").isNotNull()) &
        (col("order_delivered_carrier_date").isNull() | col("order_approved_at").isNotNull()) &
        (col("order_delivered_customer_date").isNull() | col("order_delivered_carrier_date").isNotNull()) &

        (col("order_approved_at").isNull() | (col("order_approved_at") >= col("order_purchase_timestamp"))) &
        (col("order_delivered_carrier_date").isNull() | (col("order_delivered_carrier_date") >= col("order_approved_at"))) &
        (col("order_delivered_customer_date").isNull() | (col("order_delivered_customer_date") >= col("order_delivered_carrier_date")))
    ),1
    ).otherwise(0)
)
    .select(
        "oi.order_id",
        "oi.order_item_id",
        "o.customer_unique_id",
        "oi.product_id",
        "oi.seller_id",
        "oi.price",
        "oi.freight_value",
        "oi.shipping_limit_date",
        "is_price_missing",
        "is_price_valid",
        "is_delivered_price_missing",
        "is_delivered_freight_value_missing",
        "is_freight_value_missing",
        "is_freight_value_valid",
        "is_invalid_canceled_delivery",
        "is_order_date_valid",
        "is_refunded",
        "oi.ingestion_timestamp")
)


StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 9, Finished, Available, Finished, False)

In [8]:
from delta.tables import DeltaTable

target_table = "Olist_Silver_Lakehouse.dbo.silver_olist_order_items"

if spark.catalog.tableExists(target_table):

    delta_table = DeltaTable.forName(spark, target_table)

    delta_table.alias("t").merge(
        df3_final.alias("s"),
        "t.order_id = s.order_id and t.order_item_id = s.order_item_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    df3_final.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 10, Finished, Available, Finished, False)

#### **Transforming Table : olist_order_payments**

In [9]:
valid_payment_types = ["credit_card", "boleto", "voucher", "debit_card"]
df4_final =(
    df4.withColumn(
        "ranking",
        row_number().over(Window.partitionBy("order_id", "payment_sequential")
        .orderBy(col("ingestion_timestamp").desc()))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")
    .filter(
        (col("payment_sequential") >= 1) &
        (col("payment_type").isin(valid_payment_types)) &
        (col("payment_installments") >= 1) &
        (col("payment_value") >= 0)
        )
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
        "ingestion_timestamp"
    )
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 11, Finished, Available, Finished, False)

In [10]:
from delta.tables import DeltaTable

target_table = "Olist_Silver_Lakehouse.dbo.silver_olist_order_payments"

if spark.catalog.tableExists(target_table):

    delta_table = DeltaTable.forName(spark, target_table)

    delta_table.alias("t").merge(
        df4_final.alias("s"),
        "t.order_id = s.order_id and t.payment_sequential = s.payment_sequential"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    df4_final.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 12, Finished, Available, Finished, False)

#### **Transforming Table : olist_order_reviews**

In [11]:
df5_final = (
    df5.withColumn(
        "ranking",
        row_number().over(Window.partitionBy("review_id").orderBy(col("review_creation_date").desc())
        )
    )
    .filter(col("ranking") == 1)
    .drop("ranking")

    .filter(
        (col("review_id").isNotNull()) &
        (col("order_id").isNotNull()) &
        (col("review_creation_date").isNotNull())
    )
    .withColumn(
        "is_review_score_valid",
        when(col("review_score").between(1,5),1)
        .otherwise(0)
    )
    .withColumn(
        "is_review_with_comment",
        when((col("review_comment_title").isNotNull()) | (col("review_comment_message").isNotNull()),1)
        .otherwise(0)
    )
    .withColumn(
        "is_answer_after_review",
        when((col("review_answer_timestamp").isNotNull()) & (col("review_answer_timestamp") >= col("review_creation_date")),1)
        .otherwise(0)
    )
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "is_review_score_valid",
        "is_review_with_comment",
        "is_answer_after_review",
        "ingestion_timestamp"
    )
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 13, Finished, Available, Finished, False)

In [12]:
from delta.tables import DeltaTable

target_table = "Olist_Silver_Lakehouse.dbo.silver_olist_order_reviews"

if spark.catalog.tableExists(target_table):

    delta_table = DeltaTable.forName(spark, target_table)

    delta_table.alias("t").merge(
        df5_final.alias("s"),
        "t.review_id = s.review_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

else:
    df5_final.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 14, Finished, Available, Finished, False)

#### **Transforming Table : olist_products**

In [13]:
df7_final = (
    df7.withColumn(
        "ranking",
        row_number().over(Window.partitionBy(col("product_id")).orderBy(desc("ingestion_timestamp")))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")
    .alias("p").join(df9.alias("pc"), "product_category_name", "left")
    .withColumn(
        "is_product_category_missing",
        when(col("p.product_category_name").isNull(),1)
        .otherwise(0)
    )
    .withColumn(
    "is_product_name_length_valid",
    when(col("p.product_name_lenght") >= 0,1)
    .otherwise(0)
    )
    .withColumn(
        "is_product_weight_valid",
        when(col("p.product_weight_g") >= 0,1)
        .otherwise(0)
    )
    .withColumn(
        "is_product_photos_valid",
        when(col("p.product_photos_qty") >= 0,1)
        .otherwise(0)
    )
    .withColumn(
        "is_product_dimension_valid",
        when((col("p.product_length_cm") >= 0) & (col("p.product_height_cm") >= 0) & (col("p.product_width_cm") >= 0),1)
        .otherwise(0)
    )
    .withColumn(
        "is_category_translation_available",
        when(col("pc.product_category_name").isNull(),0)
        .otherwise(1)
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name"),
        col("p.product_name_lenght"),
        col("p.product_description_lenght"),
        col("p.product_photos_qty"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm"),
        "is_product_category_missing",
        "is_category_translation_available",
        "is_product_name_length_valid",
        "is_product_weight_valid",
        "is_product_photos_valid",
        "is_product_dimension_valid",
        col("p.ingestion_timestamp")
    )
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 15, Finished, Available, Finished, False)

In [14]:
df7_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_products")

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 16, Finished, Available, Finished, False)

#### **Transforming Table : olist_sellers**

In [18]:
def normalize_column(column):
    
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")
    column = regexp_replace(column, "[-/].*", "")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df8_clean = (
    df8
    .withColumn("seller_state_clean", trim(upper(col("seller_state"))))
    .withColumn("seller_city_clean", normalize_column(col("seller_city")))
)

df10_clean = (
    df10
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("state_name_clean", normalize_column(col("state_name")))
)

df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

ref_city = df11_clean.select(col("city_name_clean")).distinct()

ref_state = df10_clean.select(col("state_code")).distinct()

ref_state_city = df11_clean.select("state_code", "city_name_clean").distinct()

df8_final = (
    df8_clean.alias("s")
    .withColumn(
        "ranking",
        row_number().over(Window.partitionBy(col("seller_id")).orderBy(desc("ingestion_timestamp")))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")
    .join(
        ref_city.alias("city_ref"),
        col("city_ref.city_name_clean") == col("s.seller_city_clean"),
        "left"
    )
    .join(
        ref_state.alias("state_ref"),
        col("state_ref.state_code") == col("s.seller_state_clean"),
        "left"
    )
    .join(
        ref_state_city.alias("sc_ref"),
       (col("s.seller_state_clean") == col("sc_ref.state_code")) &
       (col("s.seller_city_clean") == col("sc_ref.city_name_clean")),
       "left"
    )
    .withColumn(
        "is_city_valid",
        when(col("city_ref.city_name_clean").isNull(),0)
        .otherwise(1)
    )
    .withColumn(
        "is_state_valid",
        when(col("state_ref.state_code").isNull(),0)
        .otherwise(1)
    )
    .withColumn(
        "is_state_city_valid",
        when(col("sc_ref.state_code").isNull(),0)
        .otherwise(1)
    )
    .select(
        col("s.seller_id"),
        col("s.seller_zip_code_prefix"),
        col("s.seller_city"),
        col("s.seller_state"),
        "is_city_valid",
        "is_state_valid",
        "is_state_city_valid",
        col("s.ingestion_timestamp")
    )
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, 20, Finished, Available, Finished, False)

In [ ]:
df8_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_sellers")

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, -1, Cancelled, , Cancelled, True)

#### **Transforming Table : olist_product_category_translation**

In [ ]:
df9_final = (
    df9.filter(col("product_category_name").isNotNull() & col("product_category_name_english").isNotNull())
    .select(
        trim(lower(col("product_category_name"))).alias("product_category_name"),
        trim(lower(col("product_category_name_english"))).alias("product_category_name_english"),
        col("ingestion_timestamp")
    )
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, -1, Cancelled, , Cancelled, True)

In [ ]:
df9_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_product_category_translation")

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, -1, Cancelled, , Cancelled, True)

#### **Transforming Table : olist_geolocation**

In [ ]:
def normalize_column(column):
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")
    column = regexp_replace(column, "[-/].*", "")
    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df2_clean = (
    df2
    .withColumn("geolocation_state_clean", trim(upper(col("geolocation_state"))))
    .withColumn("geolocation_city_clean", normalize_column(col("geolocation_city")))
)

df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

ref_city = df11_clean.select("city_name_clean").distinct()

ref_state = df11_clean.select("state_code").distinct()  

ref_state_city = df11_clean.select(
    "state_code",
    "city_name_clean"
).distinct()

df2_final = (
    df2_clean.alias("g")
    .join(
        ref_city.alias("city_ref"),
        col("g.geolocation_city_clean") == col("city_ref.city_name_clean"),  
        "left"
    )
    .join(
        ref_state.alias("state_ref"),
        col("g.geolocation_state_clean") == col("state_ref.state_code"),  
        "left"
    )
    .join(
        ref_state_city.alias("sc_ref"),
        (col("g.geolocation_state_clean") == col("sc_ref.state_code")) &  
        (col("g.geolocation_city_clean") == col("sc_ref.city_name_clean")),  
        "left"
    )
    .withColumn(
        "is_city_valid",
        when(col("city_ref.city_name_clean").isNull(), 0).otherwise(1)
    )
    .withColumn(
        "is_state_valid",
        when(col("state_ref.state_code").isNull(), 0).otherwise(1)
    )
    .withColumn(
        "is_state_city_valid",
        when(col("sc_ref.state_code").isNull(), 0).otherwise(1)
    )
    .select(
        col("g.geolocation_zip_code_prefix"),
        col("g.geolocation_lat"),
        col("g.geolocation_lng"),
        col("g.geolocation_city"),
        col("g.geolocation_state"),
        col("is_city_valid"),          
        col("is_state_valid"),         
        col("is_state_city_valid"),    
        col("g.ingestion_timestamp")
    )
)

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, -1, Cancelled, , Cancelled, True)

In [ ]:
df2_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_geolocation")

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, -1, Cancelled, , Cancelled, True)

In [ ]:
%%sql
select count(*) from olist_customers

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, -1, Cancelled, , Cancelled, True)

In [ ]:
%%sql
select 
order_id,
payment_sequential,
count(order_id,payment_sequential) as count
from Olist_Silver_Lakehouse.silver_olist_order_payments
group by order_id, payment_sequential
having count > 1

StatementMeta(, 3f7cef18-efe5-4300-b06a-bc1d0069851c, -1, Cancelled, , Cancelled, True)